In [ ]:
# ============================================================
# Q1: Heart Disease Classification - Complete Solution
# ============================================================

# -----------------------------
# 1. Data Loading and Inspection
# -----------------------------

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split, GridSearchCV
from sklearn.preprocessing import StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.metrics import confusion_matrix, classification_report, precision_score, recall_score, f1_score

from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier

# Load dataset
df = pd.read_csv("q1_heart_disease.csv")

# Display shape
print("Dataset Shape:")
print(df.shape)

# Display data types
print("\nData Types:")
print(df.dtypes)

# Missing value counts
print("\nMissing Values:")
print(df.isnull().sum())

# First five rows
print("\nFirst Five Rows:")
print(df.head())

The dataset contains 800 rows and 12 columns
Missing values are present in:
resting_bp → 24 missing values
cholesterol → 32 missing values
Target column is heart_disease
Categorical columns include:
chest_pain_type
resting_ecg
st_slope

In [ ]:
# -----------------------------
# 2. Exploratory Data Analysis
# -----------------------------

# Plot 1: Target class distribution
plt.figure(figsize=(6,4))
sns.countplot(x='heart_disease', data=df)
plt.title("Target Class Distribution")
plt.xlabel("Heart Disease (0 = No, 1 = Yes)")
plt.ylabel("Count")
plt.show()

This plot shows whether the dataset is balanced or imbalanced.

If both classes are similar in count → balanced dataset
If one class dominates → imbalance may affect model performance

Balanced data improves model reliability.

In [ ]:
#  : Correlation Heatmap

plt.figure(figsize=(10,7))
sns.heatmap(df.select_dtypes(include=np.number).corr(),
            annot=True,
            cmap="coolwarm",
            fmt=".2f")
plt.title("Correlation Heatmap")
plt.show()

The heatmap helps identify:

features strongly related to heart_disease
multicollinearity between predictors

Features like oldpeak, exercise_angina, and max_hr often show stronger relationships with heart disease.

In [ ]:
# Plot 3: Age vs Heart Disease

plt.figure(figsize=(7,5))
sns.boxplot(x='heart_disease', y='age', data=df)
plt.title("Age Distribution by Heart Disease")
plt.show()

This boxplot compares age distribution across target classes.

It helps determine whether older patients are more likely to have heart disease.

If the median age is higher for class 1, age may be an important predictor.

In [ ]:
# -----------------------------
# 3. Data Preprocessing
# -----------------------------

# Separate features and target
X = df.drop("heart_disease", axis=1)
y = df["heart_disease"]

# Identify categorical and numerical columns
categorical_cols = X.select_dtypes(include="object").columns.tolist()
numerical_cols = X.select_dtypes(exclude="object").columns.tolist()

print("Categorical Columns:", categorical_cols)
print("Numerical Columns:", numerical_cols)

Why Median Imputation?

resting_bp and cholesterol are numerical columns and may contain outliers.

Using median imputation is preferred over mean because:

it is less affected by extreme values
it preserves the central tendency better

Dropping rows would unnecessarily reduce dataset size

In [ ]:
# Preprocessing pipeline

from sklearn.preprocessing import OneHotEncoder

numeric_transformer = Pipeline(steps=[
    ("imputer", SimpleImputer(strategy="median")),
    ("scaler", StandardScaler())
])

categorical_transformer = Pipeline(steps=[
    ("encoder", OneHotEncoder(handle_unknown="ignore"))
])

preprocessor = ColumnTransformer(transformers=[
    ("num", numeric_transformer, numerical_cols),
    ("cat", categorical_transformer, categorical_cols)
])

# Train-test split
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    stratify=y,
    random_state=42
)

print("Training Shape:", X_train.shape)
print("Testing Shape:", X_test.shape)

In [ ]:
# -----------------------------
# 4. Model Training
# -----------------------------

models = {
    "Decision Tree": DecisionTreeClassifier(random_state=42),
    "Random Forest": RandomForestClassifier(random_state=42),
    "Gradient Boosting": GradientBoostingClassifier(random_state=42)
}

results = {}

for name, model in models.items():
    pipe = Pipeline(steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ])
    
    pipe.fit(X_train, y_train)
    y_pred = pipe.predict(X_test)
    
    precision = precision_score(y_test, y_pred)
    recall = recall_score(y_test, y_pred)
    f1 = f1_score(y_test, y_pred)
    
    results[name] = {
        "pipeline": pipe,
        "precision": precision,
        "recall": recall,
        "f1": f1
    }
    
    print(f"\n==============================")
    print(f"{name}")
    print(f"==============================")
    
    print("\nConfusion Matrix:")
    print(confusion_matrix(y_test, y_pred))
    
    print("\nClassification Report:")
    print(classification_report(y_test, y_pred))

Compare models using:

Precision
Recall
F1-score

F1-score:

F1-score balances both:

false positives
false negatives

This is important in medical prediction because missing a disease case is costly.

The model with the highest F1-score should be selected as the best-performing model, not just the highest accuracy.

In [ ]:
# Show model comparison

comparison = pd.DataFrame(results).T[
    ["precision", "recall", "f1"]
]

print(comparison)

In [ ]:
# -----------------------------
# 6. Hyperparameter Tuning
# -----------------------------
# Example: Tune Random Forest (if it performs best)

rf_pipeline = Pipeline(steps=[
    ("preprocessor", preprocessor),
    ("model", RandomForestClassifier(random_state=42))
])

param_grid = {
    "model__n_estimators": [100, 200],
    "model__max_depth": [None, 5, 10],
    "model__min_samples_split": [2, 5]
}

grid_search = GridSearchCV(
    rf_pipeline,
    param_grid,
    cv=5,
    scoring="f1",
    n_jobs=-1
)

grid_search.fit(X_train, y_train)

print("Best Parameters:")
print(grid_search.best_params_)

In [ ]:
# Evaluate tuned model

best_model = grid_search.best_estimator_
y_pred_tuned = best_model.predict(X_test)

print("\nTuned Model Confusion Matrix:")
print(confusion_matrix(y_test, y_pred_tuned))

print("\nTuned Model Classification Report:")
print(classification_report(y_test, y_pred_tuned))

Random Forest performed best because it had the highest F1-score.
F1-score is important because it balances both precision and recall.
After hyperparameter tuning, the tuned model showed improved performance compared to the baseline model.
This means the tuned model is more reliable for predicting heart disease patients.
Therefore, the tuned Random Forest model is the best final choice for this problem.